# Walmart M5 Data Understanding

## Phase 1 — Data Engineering

### Milestone 1.4 — Exploratory Data Analysis (EDA)

## Business Objective

Before building a data warehouse or creating ETL pipelines, it is important to understand the structure, quality, and relationships within the Walmart M5 dataset.

The purpose of this notebook is to explore each dataset, identify potential data quality issues, and document key characteristics that will guide later database design and feature engineering.

## Import Libraries

The following libraries are used throughout this notebook for loading and inspecting the datasets.

In [1]:
import pandas as pd
import numpy as np

## Load Datasets

The Walmart M5 competition provides five primary CSV files.

Each file serves a different purpose within the forecasting workflow.

This section loads each dataset into a pandas DataFrame for exploration.

In [2]:
calendar = pd.read_csv("../data/raw/calendar.csv")

prices = pd.read_csv("../data/raw/sell_prices.csv")

sales = pd.read_csv("../data/raw/sales_train_validation.csv")

evaluation = pd.read_csv("../data/raw/sales_train_evaluation.csv")

submission = pd.read_csv("../data/raw/sample_submission.csv")

## Dataset Overview

The first step is to understand the overall size of each dataset.

Knowing the number of rows and columns helps estimate storage requirements and computational complexity.

In [3]:
print("Calendar:", calendar.shape)
print("Sell Prices:", prices.shape)
print("Sales Validation:", sales.shape)
print("Sales Evaluation:", evaluation.shape)
print("Sample Submission:", submission.shape)

Calendar: (1969, 14)
Sell Prices: (6841121, 4)
Sales Validation: (30490, 1919)
Sales Evaluation: (30490, 1947)
Sample Submission: (60980, 29)


### Observation

The datasets vary significantly in size. The sales datasets are much larger than the supporting datasets because they contain daily historical sales for thousands of product-store combinations.

## Preview the Dataset

Inspecting the first few records provides an initial understanding of the dataset's structure and column contents.

### Calendar Dataset

In [ ]:
calendar.head()

,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,HOBBIES_1_002_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,HOBBIES_1_004_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,HOBBIES_1_005_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
### Sell Prices Dataset

In [ ]:
prices.head()

### Sales Validation Dataset

In [ ]:
sales.head()

### Sales Evaluation Dataset

In [ ]:
evaluation.head()

### Sample Submission Dataset

In [ ]:
submission.head()

## Examine Data Types

Understanding the data types helps identify which columns represent numerical values, categorical variables, dates, or identifiers.

### Sell Prices Information

In [5]:
prices.info()

<class 'pandas.DataFrame'>
RangeIndex: 6841121 entries, 0 to 6841120
Data columns (total 4 columns):
 #   Column      Dtype  
---  ------      -----  
 0   store_id    str    
 1   item_id     str    
 2   wm_yr_wk    int64  
 3   sell_price  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 208.8 MB


### Sales Validation Information

In [6]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Columns: 1919 entries, id to d_1913
dtypes: int64(1913), str(6)
memory usage: 446.4 MB


## Evaluation Information

In [7]:
evaluation.info()

<class 'pandas.DataFrame'>
RangeIndex: 30490 entries, 0 to 30489
Columns: 1947 entries, id to d_1941
dtypes: int64(1941), str(6)
memory usage: 452.9 MB


## Submission Information

In [8]:
submission.info()

<class 'pandas.DataFrame'>
RangeIndex: 60980 entries, 0 to 60979
Data columns (total 29 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      60980 non-null  str  
 1   F1      60980 non-null  int64
 2   F2      60980 non-null  int64
 3   F3      60980 non-null  int64
 4   F4      60980 non-null  int64
 5   F5      60980 non-null  int64
 6   F6      60980 non-null  int64
 7   F7      60980 non-null  int64
 8   F8      60980 non-null  int64
 9   F9      60980 non-null  int64
 10  F10     60980 non-null  int64
 11  F11     60980 non-null  int64
 12  F12     60980 non-null  int64
 13  F13     60980 non-null  int64
 14  F14     60980 non-null  int64
 15  F15     60980 non-null  int64
 16  F16     60980 non-null  int64
 17  F17     60980 non-null  int64
 18  F18     60980 non-null  int64
 19  F19     60980 non-null  int64
 20  F20     60980 non-null  int64
 21  F21     60980 non-null  int64
 22  F22     60980 non-null  int64
 23  F23     60980 non-null

## Missing Value Analysis

Missing values are common in real-world datasets.

This section identifies which variables contain missing data and how frequently those missing values occur.

In [9]:
datasets = {
    "calendar": calendar,
    "prices": prices,
    "sales": sales,
    "evaluation": evaluation,
    "submission": submission
}

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(df.isnull().sum()[df.isnull().sum() > 0])


CALENDAR
event_name_1    1807
event_type_1    1807
event_name_2    1964
event_type_2    1964
dtype: int64

PRICES
Series([], dtype: int64)

SALES
Series([], dtype: int64)

EVALUATION
Series([], dtype: int64)

SUBMISSION
Series([], dtype: int64)


### Observation

The only missing values appear in the calendar event columns. This is expected because most dates do not correspond to holidays or special events. The remaining datasets contain complete records with no missing values.

## Duplicate Record Analysis

Duplicate rows can negatively affect downstream analytics and machine learning models.

This section checks whether duplicate records exist.

In [10]:
for name, df in datasets.items():
    print(name, df.duplicated().sum())

calendar 0
prices 0
sales 0
evaluation 0
submission 0


### Observation

No duplicate rows were identified across any of the five datasets, indicating that the source files are internally consistent prior to ETL processing.

## Unique Value Analysis

Understanding the number of unique categories provides insight into the dimensionality of the dataset.

In [11]:
print("Stores:", sales["store_id"].nunique())

print("Products:", sales["item_id"].nunique())

print("Departments:", sales["dept_id"].nunique())

print("Categories:", sales["cat_id"].nunique())

print("States:", sales["state_id"].nunique())

Stores: 10
Products: 3049
Departments: 7
Categories: 3
States: 3


### Observation

The Walmart M5 dataset contains:

- 10 stores
- 3 states
- 3 product categories
- 7 departments
- 3,049 unique products

These identifiers will become important dimensions within the PostgreSQL warehouse.

## Memory Usage

The Walmart M5 dataset contains millions of observations.

This section measures the approximate memory consumption of the loaded datasets.

In [12]:
for name, df in datasets.items():
    print(
        name,
        round(df.memory_usage(deep=True).sum()/1024/1024,2),
        "MB"
    )

calendar 0.67 MB
prices 853.13 MB
sales 455.38 MB
evaluation 461.9 MB
submission 17.55 MB


### Observation

The historical sales tables consume the majority of memory because every day is stored as a separate column. During ETL, these wide tables will be transformed into a normalized daily transaction format for more efficient analytics and machine learning.

## Dataset Relationships

In [13]:
print("Calendar Weeks:", calendar["wm_yr_wk"].nunique())

print("Price Weeks:", prices["wm_yr_wk"].nunique())

print("Stores:", sales["store_id"].nunique())

print("Products:", sales["item_id"].nunique())

Calendar Weeks: 282
Price Weeks: 282
Stores: 10
Products: 3049


### Observation

The Walmart M5 dataset follows a relational design.

- Calendar links dates to Walmart weeks.
- Sell Prices links products, stores, and weeks.
- Sales contains historical demand.
- Together, these files provide all information needed for demand forecasting.

## Initial Observations

Key findings from the exploratory analysis include:

- The Walmart M5 dataset contains over 30,000 product-store combinations.
- Historical sales span nearly six years of daily observations.
- Pricing information is maintained separately from sales records.
- Calendar data enriches sales with holidays, SNAP events, and weekdays.
- The dataset is highly normalized, making it well suited for PostgreSQL.
- The sales tables are stored in a wide format and will require transformation during the ETL process.
- The data appears clean, with no duplicate records and only expected missing values.

## Next Steps

The next notebook will focus on business analysis of Walmart sales by examining products, stores, departments, categories, pricing trends, and seasonality.

The insights gained will inform the PostgreSQL warehouse design and the feature engineering process used later for machine learning and revenue optimization.